# L1 — Audit and verification

Run **Runtime → Run all**. A compact package-status summary appears first; the complete verification transcript is available underneath as an expandable report.


In [ ]:
# @title Refresh repository and report tools { display-mode: "form" }
from pathlib import Path
import html as html_lib
import os
import subprocess
import sys

from IPython.display import HTML, display

repo_name = "flipkart-wired-x-campus-node"
cwd = Path.cwd()
if (cwd / "Model").is_dir() and (cwd / "requirements.txt").exists():
    repo = cwd
else:
    base = Path("/content") if Path("/content").exists() else cwd
    repo = base / repo_name
    if (repo / ".git").exists():
        subprocess.run(["git", "-C", str(repo), "pull", "--ff-only", "-q"], check=True)
    else:
        subprocess.run(
            ["git", "clone", "-q", "https://github.com/mba25015-maker/flipkart-wired-x-campus-node.git", str(repo)],
            check=True,
        )

os.chdir(repo)
if os.environ.get("WIRED_SKIP_INSTALL") != "1":
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)

model_dir = str((repo / "Model").resolve())
if model_dir not in sys.path:
    sys.path.insert(0, model_dir)

def _esc(value):
    return html_lib.escape(str(value))

def show_metrics(metrics):
    cards = []
    for label, value, note in metrics:
        cards.append(
            "<div style='background:#10265c;color:#ffffff;border:1px solid #27457f;"
            "padding:14px 16px;border-radius:10px;min-height:92px'>"
            f"<div style='font-size:12px;font-weight:700;letter-spacing:.03em;color:#dbeafe'>{_esc(label)}</div>"
            f"<div style='font-size:25px;font-weight:800;margin:5px 0 2px'>{_esc(value)}</div>"
            f"<div style='font-size:12px;color:#e4ebf7;line-height:1.35'>{_esc(note)}</div></div>"
        )
    display(HTML(
        "<div style='display:grid;grid-template-columns:repeat(auto-fit,minmax(180px,1fr));"
        "gap:12px;margin:8px 0 18px;font-family:Arial,sans-serif'>" + "".join(cards) + "</div>"
    ))

def show_table(headers, rows):
    head = "".join(
        f"<th style='background:#10265c;color:#ffffff;padding:8px 10px;text-align:left;"
        f"border:1px solid #cbd5e1'>{_esc(h)}</th>" for h in headers
    )
    body = []
    for index, row in enumerate(rows):
        background = "#ffffff" if index % 2 == 0 else "#eef3f9"
        cells = "".join(
            f"<td style='background:{background};color:#0b1f3a;padding:8px 10px;"
            f"border:1px solid #cbd5e1;vertical-align:top'>{_esc(v)}</td>" for v in row
        )
        body.append(f"<tr>{cells}</tr>")
    display(HTML(
        "<div style='overflow-x:auto;margin:6px 0 16px'>"
        "<table style='border-collapse:collapse;width:100%;font-family:Arial,sans-serif;font-size:13px'>"
        f"<thead><tr>{head}</tr></thead><tbody>{''.join(body)}</tbody></table></div>"
    ))

def show_callout(text, tone="blue"):
    palette = {
        "blue": ("#e4ebf7", "#0b1f3a", "#0070c0"),
        "green": ("#e8f5ec", "#14532d", "#157347"),
        "amber": ("#fff4cc", "#5c4300", "#ffc220"),
        "red": ("#fdecec", "#7f1d1d", "#b3261e"),
    }
    bg, fg, rule = palette[tone]
    display(HTML(
        f"<div style='background:{bg};color:{fg};border-left:6px solid {rule};padding:12px 14px;"
        f"margin:6px 0 16px;font-family:Arial,sans-serif;line-height:1.45'><b>Management read:</b> {_esc(text)}</div>"
    ))

def run_text_report(module_path):
    result = subprocess.run([sys.executable, module_path], capture_output=True, text=True)
    output = (result.stdout or "") + (result.stderr or "")
    return result.returncode, output

def show_reports(reports):
    sections = []
    failures = []
    for title, module_path in reports:
        returncode, output = run_text_report(module_path)
        if returncode:
            failures.append((title, returncode))
        sections.append(
            "<details style='margin:10px 0;border:1px solid #94a3b8;border-radius:8px;overflow:hidden'>"
            f"<summary style='cursor:pointer;font-weight:700;background:#e4ebf7;color:#0b1f3a;"
            f"padding:11px 13px;font-family:Arial,sans-serif'>Show full model report — {_esc(title)}</summary>"
            f"<pre style='white-space:pre-wrap;overflow:auto;margin:0;background:#0b1f3a !important;"
            f"color:#f8fafc !important;padding:14px;font-size:12px;line-height:1.45;"
            f"font-family:ui-monospace,SFMono-Regular,Menlo,Consolas,monospace'>{_esc(output)}</pre></details>"
        )
    display(HTML("".join(sections)))
    if failures:
        raise RuntimeError("Report failed: " + ", ".join(f"{name} (exit {code})" for name, code in failures))

print(f"Repository ready: {repo}")


In [ ]:
# @title Refresh compact management summary { display-mode: "form" }
import check_counts as counts

returncode, verification_report = run_text_report("Model/run_all.py")
package_passed = returncode == 0

show_metrics([
    ("Package status", "ALL LAYERS PASS" if package_passed else "CHECK FAILED", "One command across the complete public package"),
    ("Model checks", f"{counts.DEFINED['audit']}", "Cross-module reconciliation and claim checks"),
    ("Deck checks", f"{counts.DEFINED['deck']}", "Text, native charts, links and absence checks"),
    ("Verification layers", f"{len(counts.DEFINED)}", "Model, documents, specification, deck and artefacts"),
])
show_table(
    ["Layer", "Checks defined", "What it protects"],
    [
        ["audit.py", counts.DEFINED["audit"], "The model against itself"],
        ["verify_docs.py", counts.DEFINED["docs"], "HANDOFF figures"],
        ["verify_spec.py", counts.DEFINED["spec"], "Deck specification"],
        ["verify_deck.py", counts.DEFINED["deck"], "The exact PowerPoint"],
        ["verify_artifacts.py", counts.DEFINED["artefacts"], "Workbook, notebooks, manifest and public data"],
    ],
)
show_callout(
    "A pass establishes computational reproducibility and internal consistency; it does not independently certify every third-party source or redistribution licence.",
    "green" if package_passed else "red",
)


## How to read the summary

Every layer must pass in the same run. If the package status is red, open the full transcript below—the notebook does not hide the failing layer.


In [ ]:
# @title Show complete verification transcript { display-mode: "form" }
display(HTML(
    "<details style='margin:10px 0;border:1px solid #94a3b8;border-radius:8px;overflow:hidden'>"
    "<summary style='cursor:pointer;font-weight:700;background:#e4ebf7;color:#0b1f3a;"
    "padding:11px 13px;font-family:Arial,sans-serif'>Show full model report — package verification</summary>"
    f"<pre style='white-space:pre-wrap;overflow:auto;margin:0;background:#0b1f3a !important;"
    f"color:#f8fafc !important;padding:14px;font-size:12px;line-height:1.45;"
    f"font-family:ui-monospace,SFMono-Regular,Menlo,Consolas,monospace'>{_esc(verification_report)}</pre></details>"
))
if not package_passed:
    raise RuntimeError("Package verification failed; open the report above for the failing layer.")


## Open the model and workbook


- [Open the live model workbook in Google Sheets](https://docs.google.com/spreadsheets/d/1CEo9XOH5WeR8ZwHS0qF-ulBTGm0GlhUfNL8YzRGmyfM/edit?gid=2112744891#gid=2112744891)
- [Download `Campus_Store_Model.xlsx` from GitHub](https://github.com/mba25015-maker/flipkart-wired-x-campus-node/blob/main/Campus_Store_Model.xlsx?raw=1)
- [Browse the complete public `Model/` folder](https://github.com/mba25015-maker/flipkart-wired-x-campus-node/tree/main/Model)

- [Verification entrypoint](https://github.com/mba25015-maker/flipkart-wired-x-campus-node/blob/main/Model/run_all.py)
